# 4주차 B4 · UCI HAR Depthwise Inception-ResNet 1D

B1의 Inception-ResNet 1D에서 **Inception branch의 kernel 3·7·15 Conv1D만 `SeparableConv1D`(depthwise + pointwise)로 교체**합니다.

이번 실험의 핵심 질문은 다음과 같습니다.

> B1을 depthwise separable convolution으로 경량화하면 정확도를 얼마나 잃고, 파라미터·학습 시간·추론 시간을 얼마나 아끼는가?

B4는 정확도를 올리기 위한 실험이 아니라 **정확도와 효율의 trade-off**를 보는 실험입니다. 따라서 accuracy만큼 `params`, `train sec`, `ms/sample`을 중요하게 기록합니다.

핵심 비교는 `B1(일반 Conv)`와 `B4(Separable Conv)`입니다. 데이터 분할, 블록 수, kernel 크기, 채널 수, residual scale, 학습 조건은 같고 branch의 합성곱 연산만 다릅니다.

**실행 상태:** 이 파일은 Colab 실행 전 실험 설계본입니다. 저장된 점수를 실제 결과로 간주하지 않습니다.

**중요:** 모델 선택은 피험자 단위 validation으로 수행하고, 공식 test는 최종 구조를 정한 뒤 한 번만 확인합니다.


## 먼저 확인할 가설: 얼마나 아끼고, 얼마나 잃는가?

일반 Conv1D는 시간 패턴 추출과 채널 결합을 한 번에 합니다. SeparableConv1D는 이를 둘로 나눕니다.

1. **Depthwise:** 입력 채널마다 따로 시간 필터를 적용합니다.
2. **Pointwise:** kernel 1로 채널을 섞어 출력 채널을 만듭니다.

가설은 다음과 같습니다.

- branch 합성곱의 파라미터가 크게 줄어 **전체 파라미터가 약 40% 감소**할 것입니다(아래 손계산).
- 채널별 시간 필터와 채널 혼합을 분리하므로 표현력이 줄어 **정확도는 B1과 같거나 약간 낮을** 수 있습니다.
- 다만 이 모델은 원래 작고(약 4.8만 개) GPU에서 SeparableConv1D가 일반 Conv1D만큼 최적화되어 있지 않을 수 있으므로, **파라미터 감소가 곧바로 추론 시간 감소로 이어진다고 가정하지 않습니다.**

B4도 SIT/STAND를 직접 해결하려는 구조가 아닙니다. SIT/STAND 지표는 경량화로 인해 나빠지지 않았는지 확인하는 용도로 봅니다.


## 0. B4 구조와 비교 원칙

```text
입력 x
 ├─ bottleneck 1×1 → SeparableConv1D(3)    ← B4 변경
 ├─ bottleneck 1×1 → SeparableConv1D(7)    ← B4 변경
 ├─ bottleneck 1×1 → SeparableConv1D(15)   ← B4 변경
 └─ MaxPool(3)      → Conv1D(1)
             ↓ Concatenate
          projection 1×1
             ↓ BatchNormalization → residual scaling α=0.2
          Add([x, αF(x)])
             ↓ ReLU
```

**바꾸지 않는 것:** bottleneck, pool projection, residual projection, shortcut projection, channel_up은 모두 kernel 1(1×1)입니다. 1×1 Conv1D는 이미 pointwise 연산이므로 separable로 바꿀 이유가 없습니다. stem Conv1D(k5, 입력 9채널)도 B1과 같게 둡니다.

### 파라미터 손계산 (bias 없음)

일반 Conv1D: $K\,C_{in}\,C_{out}$, SeparableConv1D: $K\,C_{in}+C_{in}\,C_{out}$

| 블록 | branch | 입력(bottleneck) → 출력 | 일반 Conv1D | Separable |
|---|---|---|---:|---:|
| 32채널 블록 | k3 | 16 → 8 | 384 | 48 + 128 = 176 |
| | k7 | 16 → 8 | 896 | 112 + 128 = 240 |
| | k15 | 16 → 8 | 1,920 | 240 + 128 = 368 |
| | **블록 합** | | **3,200** | **784** |
| 64채널 블록 | k3 | 24 → 16 | 1,152 | 72 + 384 = 456 |
| | k7 | 24 → 16 | 2,688 | 168 + 384 = 552 |
| | k15 | 24 → 16 | 5,760 | 360 + 384 = 744 |
| | **블록 합** | | **9,600** | **1,752** |

블록이 32채널 2개, 64채널 2개이므로:

- branch 합성곱: 2×3,200 + 2×9,600 = **25,600 → 5,072** (약 80% 감소)
- 모델 전체(BN 통계 포함): B1 약 **47,846 → B4 약 27,318** (약 43% 감소)

**주의:** bottleneck이 16·24채널로 이미 좁기 때문에, 개념 노트북의 예시(32→64, 84% 감소)보다 **줄일 수 있는 절대량이 작습니다.** 남은 파라미터의 대부분은 1×1 projection(특히 64채널 블록의 64×64)이므로, 더 줄이려면 separable이 아니라 projection 크기를 다뤄야 합니다. 실제 값은 아래 `model.count_params()`로 확인합니다.

### B1과 B4에서 고정하는 조건

| 항목 | B1 | B4 |
|---|---|---|
| Inception kernel | 3·7·15 | 3·7·15 |
| branch 합성곱 | Conv1D | **SeparableConv1D** |
| 블록·채널 | 32채널 2개 + 64채널 2개 | 동일 |
| Residual / scale | 사용 / 0.2 | 동일 |
| Attention | 없음 | 없음 |
| 학습·검증 분리와 학습 조건 | 동일 | 동일 |


## 1. 라이브러리 불러오기

In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

## 2. 실험 설정

먼저 `USE_RESIDUAL=True`, `USE_SEPARABLE=True`를 유지하고 `SEED`만 0·1·2로 바꿉니다. 이것이 B4의 정식 결과입니다.

`USE_SEPARABLE=False`로 두면 B1과 같은 모델이 됩니다. 추론 시간은 런타임마다 달라지므로, **같은 Colab 런타임에서 `USE_SEPARABLE=False`(B1)와 `True`(B4)를 연달아 실행**해 시간을 비교하는 것이 가장 공정합니다.


In [ ]:
SEED = 0
USE_RESIDUAL = True
# B4 변경: Inception branch(k3·7·15)를 SeparableConv1D로 바꿀지 정하는 플래그
USE_SEPARABLE = True
RESIDUAL_SCALE = 0.2

np.random.seed(SEED)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

EXPERIMENT_NAME = (
    f"B4-res{int(USE_RESIDUAL)}"
    f"-sep{int(USE_SEPARABLE)}"
)
print("experiment:", EXPERIMENT_NAME, "| seed:", SEED)

## 3. Google Drive와 데이터 경로 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍/2week/DATA/UCI-HAR Dataset"
assert os.path.exists(DATA_PATH), f"데이터 경로를 확인하세요: {DATA_PATH}"

## 4. 9개 센서 신호와 데이터 로더 정의

In [ ]:
signals = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]


def load_data(split):
    channels = []

    for signal in signals:
        path = f"{DATA_PATH}/{split}/Inertial Signals/{signal}_{split}.txt"
        channels.append(np.loadtxt(path))

    # (9, N, 128)이 아니라 각 (N, 128) 배열을 마지막 축에 쌓아 (N, 128, 9)로 만듭니다.
    X = np.stack(channels, axis=-1)
    y = np.loadtxt(f"{DATA_PATH}/{split}/y_{split}.txt").astype(int) - 1
    subject = np.loadtxt(f"{DATA_PATH}/{split}/subject_{split}.txt").astype(int)

    return X, y, subject


X_train_all, y_train_all, s_train_all = load_data("train")
X_test, y_test, s_test = load_data("test")

print("official train:", X_train_all.shape)
print("official test :", X_test.shape, "(최종 셀 전까지 평가하지 않음)")

## 5. 피험자 단위 validation 분리

같은 사람의 윈도우가 학습과 validation에 동시에 들어가지 않도록 피험자 5·11·17·23·29를 통째로 분리합니다. 특히 5번과 17번의 `SITTING`/`STANDING` 성능을 핵심 지표로 봅니다.

In [ ]:
VAL_SUBJECTS = [5, 11, 17, 23, 29]
FOCUS_SUBJECTS = [5, 17]

val_mask = np.isin(s_train_all, VAL_SUBJECTS)

X_val = X_train_all[val_mask]
y_val = y_train_all[val_mask]
s_val = s_train_all[val_mask]

X_train = X_train_all[~val_mask]
y_train = y_train_all[~val_mask]
s_train = s_train_all[~val_mask]

print("train:", X_train.shape, "subjects:", sorted(np.unique(s_train)))
print("val  :", X_val.shape, "subjects:", sorted(np.unique(s_val)))

## 6. 학습 데이터 통계로 정규화

In [ ]:
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = ((X_train - mean) / (std + 1e-8)).astype("float32")
X_val = ((X_val - mean) / (std + 1e-8)).astype("float32")
X_test = ((X_test - mean) / (std + 1e-8)).astype("float32")

n_outputs = 6
y_train_onehot = to_categorical(y_train, num_classes=n_outputs)
y_val_onehot = to_categorical(y_val, num_classes=n_outputs)

## 7. Depthwise Inception-residual 1D 블록

세 branch는 kernel 3·7·15로 서로 다른 시간 범위를 봅니다. B4에서는 이 세 branch만 `SeparableConv1D`로 바꿉니다.

- `SeparableConv1D`는 먼저 bottleneck 채널마다 따로 시간 필터(depthwise)를 적용하고, 다음에 1×1로 채널을 섞어(pointwise) branch 출력 채널을 만듭니다.
- 레이어 이름(`*_conv_k3` 등)은 B1과 같게 두어 shape 검사와 파라미터 비교 코드를 그대로 쓸 수 있게 했습니다.
- pooling branch와 projection은 1×1이므로 B1과 동일합니다.


In [ ]:
def inception_residual_block(
    x,
    filters,
    bottleneck_filters=16,
    kernel_sizes=(3, 7, 15),
    residual_scale=0.2,
    use_residual=True,
    use_separable=True,
    name="ir",
):
    if filters % 4 != 0:
        raise ValueError("filters는 네 branch에 나눌 수 있도록 4의 배수여야 합니다.")

    branch_filters = filters // 4
    shortcut = x

    # 큰 kernel을 적용하기 전에 채널을 줄여 계산량을 제한합니다.
    bottleneck = layers.Conv1D(
        bottleneck_filters, 1, padding="same", use_bias=False,
        name=f"{name}_bottleneck",
    )(x)
    bottleneck = layers.BatchNormalization(name=f"{name}_bottleneck_bn")(bottleneck)
    bottleneck = layers.Activation("relu", name=f"{name}_bottleneck_relu")(bottleneck)

    branches = []
    # B4 변경: kernel 3·7·15 branch만 SeparableConv1D(depthwise + pointwise)로 교체
    # 이유: 시간 필터링(채널별)과 채널 결합(1×1)을 분리해 branch 파라미터·연산량을 줄이기 위함
    #       bottleneck·pool projection·residual projection은 이미 1×1 이라 그대로 둡니다.
    if use_separable:
        branch_conv = layers.SeparableConv1D
    else:
        branch_conv = layers.Conv1D

    for kernel_size in kernel_sizes:
        branch = branch_conv(
            branch_filters, kernel_size, padding="same", use_bias=False,
            name=f"{name}_conv_k{kernel_size}",
        )(bottleneck)
        branches.append(branch)

    pool_branch = layers.MaxPooling1D(
        pool_size=3, strides=1, padding="same", name=f"{name}_pool"
    )(x)
    pool_branch = layers.Conv1D(
        branch_filters, 1, padding="same", use_bias=False,
        name=f"{name}_pool_projection",
    )(pool_branch)
    branches.append(pool_branch)

    merged = layers.Concatenate(axis=-1, name=f"{name}_concatenate")(branches)

    # Inception 결과를 shortcut과 같은 채널 수로 투영합니다.
    residual = layers.Conv1D(
        filters, 1, padding="same", use_bias=False,
        name=f"{name}_residual_projection",
    )(merged)
    residual = layers.BatchNormalization(name=f"{name}_residual_bn")(residual)

    # shortcut을 끈 대조 실험에서도 같은 배율을 적용하여
    # 비교 조건에서 shortcut 유무만 달라지게 합니다.
    residual = layers.Rescaling(
        residual_scale, name=f"{name}_residual_scale"
    )(residual)

    if use_residual:
        if x.shape[-1] != filters:
            shortcut = layers.Conv1D(
                filters, 1, padding="same", use_bias=False,
                name=f"{name}_shortcut_projection",
            )(shortcut)
            shortcut = layers.BatchNormalization(name=f"{name}_shortcut_bn")(shortcut)

        x = layers.Add(name=f"{name}_add")([shortcut, residual])
    else:
        # shortcut만 뺀 대조 실험입니다. 나머지 Inception 경로는 동일합니다.
        x = residual

    return layers.Activation("relu", name=f"{name}_out")(x)

## 8. B4 모델 만들기

Stem, 네 Inception-residual 블록, pooling, 분류 head는 B1과 같습니다. 각 블록의 branch 합성곱만 SeparableConv1D입니다.

모델을 만든 뒤 전체 파라미터와 함께 **branch 합성곱(`*_conv_k*`)만의 파라미터**를 따로 출력합니다. 위 손계산(B1 25,600 → B4 5,072)과 맞는지 확인합니다.


In [ ]:
# B4 변경: use_separable 을 받아 각 블록에 전달
def build_b4_model(use_residual=True, use_separable=True, residual_scale=0.2):
    inputs = layers.Input(shape=(128, 9), name="sensor_window")

    x = layers.Conv1D(32, 5, padding="same", use_bias=False, name="stem_conv")(inputs)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("relu", name="stem_relu")(x)

    x = inception_residual_block(
        x, filters=32, bottleneck_filters=16,
        residual_scale=residual_scale, use_residual=use_residual,
        use_separable=use_separable, name="ir32_1",
    )
    x = inception_residual_block(
        x, filters=32, bottleneck_filters=16,
        residual_scale=residual_scale, use_residual=use_residual,
        use_separable=use_separable, name="ir32_2",
    )

    x = layers.MaxPooling1D(2, name="downsample_pool")(x)
    x = layers.Conv1D(64, 1, padding="same", use_bias=False, name="channel_up")(x)
    x = layers.BatchNormalization(name="channel_up_bn")(x)
    x = layers.Activation("relu", name="channel_up_relu")(x)

    x = inception_residual_block(
        x, filters=64, bottleneck_filters=24,
        residual_scale=residual_scale, use_residual=use_residual,
        use_separable=use_separable, name="ir64_1",
    )
    x = inception_residual_block(
        x, filters=64, bottleneck_filters=24,
        residual_scale=residual_scale, use_residual=use_residual,
        use_separable=use_separable, name="ir64_2",
    )

    x = layers.GlobalAveragePooling1D(name="global_average_pooling")(x)
    x = layers.Dropout(0.4, name="classifier_dropout")(x)
    outputs = layers.Dense(n_outputs, activation="softmax", name="activity")(x)

    return Model(inputs, outputs, name="UCI_HAR_B4_Depthwise_Inception_ResNet1D")


model = build_b4_model(
    use_residual=USE_RESIDUAL,
    use_separable=USE_SEPARABLE,
    residual_scale=RESIDUAL_SCALE,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()
print(f"parameters: {model.count_params():,}")

# B4 변경: branch 합성곱만의 파라미터를 따로 확인 (손계산: B1 25,600 → B4 5,072)
branch_params = sum(
    layer.count_params() for layer in model.layers if "_conv_k" in layer.name
)
print(f"branch conv parameters: {branch_params:,}")


## 9. `Concatenate`와 `Add` shape 확인

`Concatenate` 뒤에는 여러 branch의 채널이 모이고, projection을 거친 residual과 shortcut은 같은 shape이어야 합니다. 이 검사는 잘못된 `Add` 설계를 실행 전에 발견하기 위한 것입니다.

In [ ]:
for block_name in ["ir32_1", "ir32_2", "ir64_1", "ir64_2"]:
    merged_shape = model.get_layer(f"{block_name}_concatenate").output.shape
    residual_shape = model.get_layer(f"{block_name}_residual_bn").output.shape
    output_shape = model.get_layer(f"{block_name}_out").output.shape
    print(block_name, "concat:", merged_shape, "residual:", residual_shape, "out:", output_shape)

    if USE_RESIDUAL:
        add_layer = model.get_layer(f"{block_name}_add")
        left_shape, right_shape = [tensor.shape for tensor in add_layer.input]
        assert left_shape == right_shape, f"{block_name} Add shape 불일치"

## 10. 학습

기존 S0와 마찬가지로 최대 100에폭, batch size 64, Adam과 EarlyStopping을 사용합니다. 구조 외의 조건을 가능한 한 고정하고 학습 시간도 함께 기록합니다.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=10,
        restore_best_weights=True, verbose=1,
    )
]

train_start = time.perf_counter()

history = model.fit(
    X_train,
    y_train_onehot,
    validation_data=(X_val, y_val_onehot),
    epochs=100,
    batch_size=64,
    shuffle=True,
    callbacks=callbacks,
    verbose=1,
)

training_seconds = time.perf_counter() - train_start
best_epoch = int(np.argmin(history.history["val_loss"])) + 1

print(f"training time: {training_seconds:.1f} sec")
print(f"best epoch by val_loss: {best_epoch}")

## 11. Validation 전체 평가

In [ ]:
class_names = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING", "STANDING", "LAYING",
]

y_val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
val_accuracy = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)
cm_val = confusion_matrix(y_val, y_val_pred, labels=range(n_outputs))

print(f"validation accuracy : {val_accuracy:.4f}")
print(f"validation macro F1 : {val_f1:.4f}")
print(classification_report(
    y_val, y_val_pred, target_names=class_names,
    digits=4, zero_division=0,
))
print("confusion matrix")
print(cm_val)

## 12. `SITTING`/`STANDING`과 피험자별 평가

전체 정확도가 좋아져도 기존 핵심 문제인 앉음↔섬 오류가 늘어날 수 있으므로 별도로 기록합니다.

In [ ]:
SIT, STAND = 3, 4

print("subject | all acc | SIT/STAND acc | SIT->STAND | STAND->SIT")
for subject in VAL_SUBJECTS:
    mask = s_val == subject
    ss_mask = mask & np.isin(y_val, [SIT, STAND])
    sit_to_stand = np.sum(mask & (y_val == SIT) & (y_val_pred == STAND))
    stand_to_sit = np.sum(mask & (y_val == STAND) & (y_val_pred == SIT))
    mark = " <- focus" if subject in FOCUS_SUBJECTS else ""

    print(
        f"{subject:7d} | {accuracy_score(y_val[mask], y_val_pred[mask]):.4f}  | "
        f"{accuracy_score(y_val[ss_mask], y_val_pred[ss_mask]):.4f}        | "
        f"{sit_to_stand:10d} | {stand_to_sit:10d}{mark}"
    )

focus_mask = np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])
other_mask = ~np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

focus_ss_acc = accuracy_score(y_val[focus_mask], y_val_pred[focus_mask])
other_ss_acc = accuracy_score(y_val[other_mask], y_val_pred[other_mask])
val_ss_err = int(cm_val[SIT, STAND] + cm_val[STAND, SIT])
val_dynamic_err = int(cm_val[0:3, 0:3].sum() - np.trace(cm_val[0:3, 0:3]))

## 13. 추론 시간 측정

GPU·CPU 종류에 따라 절대 시간은 달라집니다. 따라서 같은 Colab 런타임에서 다른 모델과 측정한 값끼리만 비교합니다.

In [ ]:
timing_batch = tf.convert_to_tensor(X_val[:256])

# 초기 그래프 준비와 장치 준비 시간을 제외하기 위한 warm-up입니다.
for _ in range(3):
    model(timing_batch, training=False).numpy()

repeats = 20
inference_start = time.perf_counter()
for _ in range(repeats):
    model(timing_batch, training=False).numpy()
inference_seconds = time.perf_counter() - inference_start

inference_ms_per_sample = (
    inference_seconds * 1000 / (repeats * len(timing_batch))
)
print(f"inference: {inference_ms_per_sample:.4f} ms/sample")

## 14. Gradient norm 관찰

고정된 validation mini-batch에서 loss를 한 번 계산하고 각 Conv1D kernel의 gradient norm을 출력합니다. 앞쪽 층의 gradient가 반복적으로 매우 작고 residual을 끈 실험보다도 작다면 gradient 전달 문제를 의심할 수 있습니다.

단, 한 시점의 gradient norm만으로 gradient 소실을 증명할 수는 없습니다. 동일 seed에서 `USE_SEPARABLE=True/False`를 비교하고 학습곡선·수렴 에폭·정확도를 함께 봅니다.

In [ ]:
probe_x = tf.convert_to_tensor(X_val[:64])
probe_y = tf.convert_to_tensor(y_val_onehot[:64])

with tf.GradientTape() as tape:
    probe_prediction = model(probe_x, training=False)
    probe_loss = tf.reduce_mean(
        tf.keras.losses.categorical_crossentropy(probe_y, probe_prediction)
    )

gradients = tape.gradient(probe_loss, model.trainable_variables)
gradient_rows = []

for variable, gradient in zip(model.trainable_variables, gradients):
    variable_path = getattr(variable, "path", variable.name)
    if gradient is not None and "kernel" in variable_path:
        gradient_rows.append((variable_path, float(tf.norm(gradient))))

print("loss:", float(probe_loss))
print("\n앞쪽 kernel gradient norm")
for name, norm in gradient_rows[:6]:
    print(f"{name:55s} {norm:.8f}")

print("\n뒤쪽 kernel gradient norm")
for name, norm in gradient_rows[-6:]:
    print(f"{name:55s} {norm:.8f}")

## 15. 학습곡선

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")
plt.xlabel("epoch")
plt.title(f"{EXPERIMENT_NAME} loss · seed {SEED}")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")
plt.xlabel("epoch")
plt.title(f"{EXPERIMENT_NAME} accuracy · seed {SEED}")
plt.legend()

plt.tight_layout()
plt.show()

## 16. 결과 한 줄 요약

In [ ]:
print(
    f"{EXPERIMENT_NAME} | seed {SEED} | "
    f"val acc {val_accuracy:.4f} | val F1 {val_f1:.4f} | "
    f"focus SIT/STAND(5,17) {focus_ss_acc:.4f} | "
    f"other SIT/STAND(11,23,29) {other_ss_acc:.4f} | "
    f"SIT<->STAND errors {val_ss_err} | dynamic errors {val_dynamic_err} | "
    f"epochs {len(history.history['val_loss'])} (best {best_epoch}) | "
    f"params {model.count_params():,} | train sec {training_seconds:.1f} | "
    f"inference ms/sample {inference_ms_per_sample:.4f}"
)

## 17. B1·B4 결과 기록표

B1과 B4를 같은 seed끼리 비교합니다. B4는 효율 실험이므로 **Params·Train sec·ms/sample**을 반드시 함께 적습니다. 시간 값은 같은 Colab 런타임에서 측정한 것끼리만 비교합니다.

| 실험 | seed | Branch conv | Val Accuracy | Macro F1 | Focus SIT/STAND | Dynamic errors | **Params** | **Branch params** | Best epoch | **Train sec** | **ms/sample** |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| B1 | 0 | Conv1D |  |  |  |  |  |  |  |  |  |
| B4 | 0 | Separable |  |  |  |  |  |  |  |  |  |
| B1 | 1 | Conv1D |  |  |  |  |  |  |  |  |  |
| B4 | 1 | Separable |  |  |  |  |  |  |  |  |  |
| B1 | 2 | Conv1D |  |  |  |  |  |  |  |  |  |
| B4 | 2 | Separable |  |  |  |  |  |  |  |  |  |

### 해석 기준

- 정확도 차이(`B4 − B1`)와 파라미터·시간 감소율을 **함께** 적습니다. 예: "파라미터 43% 감소, val accuracy 평균 −0.3%p".
- 세 seed의 평균·표준편차 안에 정확도 차이가 들어오면 "정확도 손실 없이 경량화했다"고 말할 수 있습니다.
- 파라미터는 줄었는데 ms/sample이 줄지 않았다면, 작은 모델에서는 SeparableConv1D의 연산 이점이 GPU 실행 속도로 이어지지 않았다는 뜻입니다. 이것도 기록할 결과입니다.
- B4가 B1보다 정확도까지 높다면 작은 데이터에서 파라미터 감소가 과적합을 줄였을 가능성을 학습곡선으로 확인합니다.


## 18. 🔒 공식 test 평가

모든 후보를 같은 validation 조건으로 비교하고 최종 모델을 결정한 뒤 한 번만 실행합니다. 기본값에서는 오류를 발생시키지 않고 “건너뜀” 메시지만 출력하므로, Colab의 “모두 실행” 결과를 깨끗하게 저장할 수 있습니다.

In [ ]:
FINAL_TEST = False

if not FINAL_TEST:
    print(
        "공식 test를 건너뜁니다. validation으로 최종 모델을 정한 뒤 "
        "이 셀의 FINAL_TEST=True로 변경해 한 번만 실행하세요."
    )
else:
    y_test_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
    cm_test = confusion_matrix(y_test, y_test_pred, labels=range(n_outputs))

    print(f"test accuracy : {test_accuracy:.4f}")
    print(f"test macro F1 : {test_f1:.4f}")
    print(classification_report(
        y_test, y_test_pred, target_names=class_names,
        digits=4, zero_division=0,
    ))
    print("confusion matrix")
    print(cm_test)

    test_ss_err = int(cm_test[SIT, STAND] + cm_test[STAND, SIT])
    test_dynamic_err = int(cm_test[0:3, 0:3].sum() - np.trace(cm_test[0:3, 0:3]))

    print(
        f"{EXPERIMENT_NAME} TEST | seed {SEED} | acc {test_accuracy:.4f} | "
        f"macro F1 {test_f1:.4f} | SIT<->STAND {test_ss_err} | "
        f"dynamic errors {test_dynamic_err}"
    )

## 19. 참고 자료

- Szegedy et al., *Inception-v4, Inception-ResNet and the Impact of Residual Connections on Learning*, 2016
- He et al., *Deep Residual Learning for Image Recognition*, 2015
- Chollet, *Xception: Deep Learning with Depthwise Separable Convolutions*, 2016
- Ismail Fawaz et al., *InceptionTime: Finding AlexNet for Time Series Classification*, 2019
- UCI Machine Learning Repository, *Human Activity Recognition Using Smartphones*